## 0. Install Required Libraries

Run once, then restart kernel.

In [1]:
%pip install pandas numpy scikit-learn pyswarm

  Using cached pyswarm-0.6.tar.gz (4.3 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for pyswarm: filename=pyswarm-0.6-py3-none-any.whl size=4540 sha256=6d949e68daf130ecc2c897eb351409cd0ea9235821b8782624a8ea56a018ec30
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\40\f3\da\00b82a03b46209e55182b57412ec1d6b21c9795ead6b048313
Successfully built pyswarm
Note: you may need to restart the kernel to use updated packages.


# Satellite Collision Avoidance
### ESA Kelvins Challenge
Source: https://kelvins.esa.int/collision-avoidance-challenge/data/

---

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from pyswarm import pso

print("All libraries imported successfully!")

All libraries imported successfully!


---
## 2. Import ESA Dataset

Download from https://kelvins.esa.int/collision-avoidance-challenge/data/
1. Register (free)
2. Download train_data.zip -> extract -> train_data.csv
3. Download test_data.csv
4. Place both in C:/Users/potha/satellite_collision/

In [2]:
train_path = "train_data.csv"
test_path  = "test_data.csv"

print("Dataset File Check:")
train_found = os.path.exists(train_path)
test_found  = os.path.exists(test_path)
print(f"  train_data.csv : {'FOUND' if train_found else 'NOT FOUND'}")
print(f"  test_data.csv  : {'FOUND' if test_found  else 'NOT FOUND'}")

if not train_found or not test_found:
    raise FileNotFoundError("Place both CSV files in the folder and re-run.")
else:
    print("Both files found. Ready to load!")

Dataset File Check:
  train_data.csv : FOUND
  test_data.csv  : FOUND
Both files found. Ready to load!


---
## 3. Load Dataset

In [3]:
train = pd.read_csv("train_data.csv")
test  = pd.read_csv("test_data.csv")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Columns     : {list(train.columns)}")

Train shape : (162634, 103)
Test shape  : (24484, 103)
Columns     : ['event_id', 'time_to_tca', 'mission_id', 'risk', 'max_risk_estimate', 'max_risk_scaling', 'miss_distance', 'relative_speed', 'relative_position_r', 'relative_position_t', 'relative_position_n', 'relative_velocity_r', 'relative_velocity_t', 'relative_velocity_n', 't_time_lastob_start', 't_time_lastob_end', 't_recommended_od_span', 't_actual_od_span', 't_obs_available', 't_obs_used', 't_residuals_accepted', 't_weighted_rms', 't_rcs_estimate', 't_cd_area_over_mass', 't_cr_area_over_mass', 't_sedr', 't_j2k_sma', 't_j2k_ecc', 't_j2k_inc', 't_ct_r', 't_cn_r', 't_cn_t', 't_crdot_r', 't_crdot_t', 't_crdot_n', 't_ctdot_r', 't_ctdot_t', 't_ctdot_n', 't_ctdot_rdot', 't_cndot_r', 't_cndot_t', 't_cndot_n', 't_cndot_rdot', 't_cndot_tdot', 'c_object_type', 'c_time_lastob_start', 'c_time_lastob_end', 'c_recommended_od_span', 'c_actual_od_span', 'c_obs_available', 'c_obs_used', 'c_residuals_accepted', 'c_weighted_rms', 'c_rcs_estimat

In [4]:
print("Risk Score Statistics:")
print(train["risk"].describe())
print(f"Zero-risk rows : {(train['risk'] == 0).sum()}")
print(f"Non-zero risk  : {(train['risk'] > 0).sum()}")
train.head(3)

Risk Score Statistics:
count    162634.000000
mean        -19.340603
std          10.011641
min         -30.000000
25%         -30.000000
50%         -17.870632
75%          -9.173294
max          -1.442854
Name: risk, dtype: float64
Zero-risk rows : 0
Non-zero risk  : 0


,event_id,time_to_tca,mission_id,risk,max_risk_estimate,max_risk_scaling,miss_distance,relative_speed,relative_position_r,relative_position_t,...,t_sigma_rdot,c_sigma_rdot,t_sigma_tdot,c_sigma_tdot,t_sigma_ndot,c_sigma_ndot,F10,F3M,SSN,AP
0,0,1.566798,5,-10.204955,-7.834756,8.602101,14923.0,13792.0,453.8,5976.6,...,0.147350,58.272095,0.004092,0.165044,0.002987,0.386462,89.0,83.0,42.0,11.0
1,0,1.207494,5,-10.355758,-7.848937,8.956374,14544.0,13792.0,474.3,5821.2,...,0.059672,57.966413,0.003753,0.164383,0.002933,0.386393,89.0,83.0,42.0,11.0
2,0,0.952193,5,-10.345631,-7.847406,8.932195,14475.0,13792.0,474.6,5796.2,...,0.039258,57.907599,0.003576,0.164352,0.002967,0.386381,89.0,83.0,42.0,11.0


---
## 4. Feature Selection

All numeric columns used. Excluded: event_id, mission_id, c_object_type, risk.

In [5]:
exclude_cols = ["risk", "event_id", "mission_id", "c_object_type"]
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
features = [col for col in numeric_cols if col not in exclude_cols]

print(f"Total features selected: {len(features)}")

X_all  = train[features].fillna(0)
y_all  = train["risk"]
X_test = test[features].fillna(0)

X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

print(f"Training samples   : {X_train.shape[0]}")
print(f"Validation samples : {X_val.shape[0]}")
print(f"Test samples       : {X_test.shape[0]}")

Total features selected: 99
Training samples   : 130107
Validation samples : 32527
Test samples       : 24484


---
## 5. Feature Scaling

StandardScaler: zero mean, unit variance. Fit on train only.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")
print(f"Train mean (first 5, ~0): {X_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Train std  (first 5, ~1): {X_train_scaled.std(axis=0)[:5].round(4)}")

Scaling complete.
Train mean (first 5, ~0): [ 0.  0. -0. -0. -0.]
Train std  (first 5, ~1): [1. 1. 1. 1. 1.]


---
## 6. Train Collision Risk Model

Random Forest - 200 trees, all CPU cores.

In [7]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training model... (may take 1-2 minutes)")
model.fit(X_train_scaled, y_train)
print("Model trained successfully!")

Training model... (may take 1-2 minutes)
Model trained successfully!


---
## 7. Evaluate on Validation Set

In [8]:
val_preds = model.predict(X_val_scaled)
mae = mean_absolute_error(y_val, val_preds)
r2  = r2_score(y_val, val_preds)

print(f"Validation MAE : {mae:.6f}  (lower is better)")
print(f"Validation R2  : {r2:.4f}   (1.0 = perfect)")

Validation MAE : 0.023199  (lower is better)
Validation R2  : 0.9999   (1.0 = perfect)


---
## 8. Predict Collision Risk on Test Set

In [9]:
predicted_risk = model.predict(X_test_scaled)
test["predicted_risk"] = predicted_risk

print("Predicted Collision Risk Summary:")
print(test["predicted_risk"].describe())
print(f"High risk  (> 0.001) : {(test['predicted_risk'] > 0.001).sum()}")
print(f"Very high  (> 0.01)  : {(test['predicted_risk'] > 0.01).sum()}")

Predicted Collision Risk Summary:
count    24484.000000
mean       -16.549895
std          9.844530
min        -30.000000
25%        -30.000000
50%        -13.547562
75%         -7.422489
max         -1.920336
Name: predicted_risk, dtype: float64
High risk  (> 0.001) : 0
Very high  (> 0.01)  : 0


In [10]:
print("Top 10 Highest Risk Events:")
test[["event_id", "predicted_risk"]].sort_values("predicted_risk", ascending=False).head(10)

Top 10 Highest Risk Events:


,event_id,predicted_risk
14267,1259,-1.920336
14266,1259,-1.937455
14268,1259,-2.333028
14265,1259,-2.358236
17716,1568,-2.425928
17714,1568,-2.472878
17717,1568,-2.480580
17713,1568,-2.501444
17711,1568,-2.519267
17715,1568,-2.527567


---
## 9. PSO - Optimal Collision Avoidance Maneuver

Finds [delta_r, delta_t, delta_n] that minimises risk for the highest-risk event.

In [11]:
target_idx    = test["predicted_risk"].idxmax()
target_pos    = test.index.get_loc(target_idx)
target_sample = X_test_scaled[target_pos].copy()

print(f"Event ID               : {test.loc[target_idx, 'event_id']}")
print(f"Current predicted risk : {test.loc[target_idx, 'predicted_risk']:.6f}")

def collision_risk_pso(maneuver):
    sample = target_sample.copy()
    sample[0] += maneuver[0]  # Radial
    sample[1] += maneuver[1]  # Tangential
    sample[2] += maneuver[2]  # Normal
    return model.predict([sample])[0]

best_maneuver, best_risk = pso(
    collision_risk_pso,
    [-10, -10, -10],
    [ 10,  10,  10],
    swarmsize=30,
    maxiter=50
)

print("PSO optimization complete!")

Event ID               : 1259
Current predicted risk : -1.920336
Stopping search: Swarm best objective change less than 1e-08
PSO optimization complete!


---
## 10. Results

In [12]:
original_risk = test.loc[target_idx, "predicted_risk"]
reduction     = ((original_risk - best_risk) / original_risk) * 100 if original_risk > 0 else 0

print("=" * 50)
print("     Optimal Collision Avoidance Maneuver")
print("=" * 50)
print(f"  Delta R (Radial)      : {best_maneuver[0]:.6f}")
print(f"  Delta T (Tangential)  : {best_maneuver[1]:.6f}")
print(f"  Delta N (Normal)      : {best_maneuver[2]:.6f}")
print("-" * 50)
print(f"  Risk BEFORE maneuver  : {original_risk:.6f}")
print(f"  Risk AFTER  maneuver  : {best_risk:.6f}")
print(f"  Risk Reduction        : {reduction:.2f}%")
print("=" * 50)

     Optimal Collision Avoidance Maneuver
  Delta R (Radial)      : 8.255273
  Delta T (Tangential)  : -10.000000
  Delta N (Normal)      : 2.165590
--------------------------------------------------
  Risk BEFORE maneuver  : -1.920336
  Risk AFTER  maneuver  : -29.876784
  Risk Reduction        : 0.00%


---
## 11. Save Results

In [13]:
output_path = "collision_risk_results.csv"
test[["event_id", "predicted_risk"]].to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
test[["event_id", "predicted_risk"]].sort_values("predicted_risk", ascending=False).head(10)

Saved to: collision_risk_results.csv


,event_id,predicted_risk
14267,1259,-1.920336
14266,1259,-1.937455
14268,1259,-2.333028
14265,1259,-2.358236
17716,1568,-2.425928
17714,1568,-2.472878
17717,1568,-2.480580
17713,1568,-2.501444
17711,1568,-2.519267
17715,1568,-2.527567
